In [19]:
import os
import torch
import numpy as np
import pandas as pd
from config import *
import torch.nn as nn
from tqdm import tqdm
from path import Path
import torch.optim.adam
from torchvision import models
from torch.utils.data import Dataset
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

In [20]:
def createDataframe(rootDir):
    rows = []
    for label in os.listdir(rootDir):
        for image in tqdm(os.listdir(f"{rootDir}{label}")):
            img = np.load(f"{rootDir}{label}/{image}")[0].astype(np.float32)
            img_log = np.log1p(img)

            rows.append({
                "file_path": f"{rootDir}{label}/{image}",
                "label": label,
                "mean": np.mean(img_log),
                "std": np.std(img_log)
            })
    return pd.DataFrame(rows)


class customDataset(Dataset):
    def __init__(self, dataframe, transform = None):
        self.df = dataframe
        self.transform = transform
        self.label_keys = {label: i for i, label in enumerate(dataframe['label'].unique())}
        self.global_mean = self.df["mean"].mean()
        self.global_std = self.df["std"].mean()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_path = self.df.iloc[idx]["file_path"]
        raw_data = np.load(file_path)[0].astype(np.float32)
        scaled_data = np.log1p(raw_data)
        data = (scaled_data - self.global_mean) / (self.global_std + 1e-7)
        data_tensor = torch.from_numpy(data).float().unsqueeze(0)
        label = self.label_keys[self.df.iloc[idx]["label"]]

        if self.transform:
            data_tensor = self.transform(data_tensor)

        return data_tensor, label

train_transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=180),
    v2.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

if not Path(BASE+"DatasetCommon/dataset/train.csv").exists():
    train_df = createDataframe(BASE+"DatasetCommon/dataset/train/")
    train_df.to_csv(BASE+"DatasetCommon/dataset/train.csv")
else:
    train_df = pd.read_csv(BASE+"DatasetCommon/dataset/train.csv")

training_data = customDataset(train_df, train_transform)

if not Path(BASE+"DatasetCommon/dataset/val.csv").exists():
    val_df = createDataframe(BASE+"DatasetCommon/dataset/val/")
    val_df.to_csv(BASE+"DatasetCommon/dataset/val.csv")
else:
    val_df = pd.read_csv(BASE+"DatasetCommon/dataset/val.csv")

val_data = customDataset(val_df)

In [21]:
# 1. Setup Data
train_loader = DataLoader(training_data, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=128, shuffle=True, num_workers=0)

# 2. Initialize ResNet-18
model = models.resnet18(weights=None)

# CHANGE: Update the first conv layer to accept 1 channel instead of 3
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# CHANGE: Update the last layer for 3 classes (Subhalo, Vortex, No_sub)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3)

model = model.to('cuda') # Move to GPU!

# 3. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [23]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Get probabilities for AUC (Softmax converts logits to 0-1 range)
            probs = torch.softmax(outputs, dim=1)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            running_loss += loss.item() * images.size(0)

    # Prepare data for Scikit-Learn
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Calculate Multiclass AUC (One-vs-Rest)
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    avg_loss = running_loss / len(loader.dataset)

    return avg_loss, auc

# --- Execution ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load(BASE+"DatasetCommon/model.pt"))
model.to(device)

for epoch in range(10, 50): # Start with 10 to see progress
    print("Epoch:", epoch+1)
    print("Training: ")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    print("Validation: ")
    val_loss, val_auc = validate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")

torch.save(model.state_dict(), BASE+"DatasetCommon/model.pt")

Epoch: 11
Training: 


100%|██████████| 235/235 [02:06<00:00,  1.86it/s]


Validation: 


100%|██████████| 59/59 [00:10<00:00,  5.68it/s]


Epoch 11: Train Loss: 0.8645 | Val Loss: 0.8565 | Val AUC: 0.7949
Epoch: 12
Training: 


...

Epoch 49: Train Loss: 0.2754 | Val Loss: 0.3008 | Val AUC: 0.9792
Epoch: 50
Training: 


100%|██████████| 235/235 [01:59<00:00,  1.97it/s]


Validation: 


100%|██████████| 59/59 [00:09<00:00,  6.34it/s]

Epoch 50: Train Loss: 0.2699 | Val Loss: 0.3105 | Val AUC: 0.9780
